In [ ]:
import numpy as np
from feos.eos import EquationOfState, PhaseDiagram
from feos.pets import PetsParameters
import feos.si as si

import matplotlib.pyplot as plt
import pandas as pd
import keras

import sys
sys.path.append("..")
import maxwell_construct as maxwell
from LJEOS import calc_mu

from scipy.integrate import simpson
from scipy.optimize import curve_fit

In [2]:
def generate_windows(array, bins):
    """
    Generate sliding windows for the input array with a given bin size.

    Parameters:
    - array (np.ndarray): Input array.
    - bins (int): Number of bins on each side of the central bin.
    - mode (str): Padding mode for np.pad (default is "wrap").

    Returns:
    - np.ndarray: Array of sliding windows.
    """
    padded_array = np.pad(array, bins, mode="wrap")
    windows = np.empty((len(array), 2 * bins + 1))
    for i in range(len(array)):
        windows[i] = padded_array[i:i + 2 * bins + 1]
    return windows


def c1_pred(model, rho, T, input_bins=1201):
    
    T = T * np.ones_like(rho)

    window_bins = (input_bins - 1) // 2
    rho_windows = generate_windows(rho, window_bins).reshape(rho.shape[0], input_bins, 1)
    
    c1_result = model.predict_on_batch([rho_windows, T]).flatten()
    return c1_result 

def c1_pred_lr(model, rho, T, dx=0.005, input_bins=1201, rc = 2.5, eos="pets"):
    # for a bulk system delta phi is zero so this has been neglected here.
    
    T = T * np.ones_like(rho)

    window_bins = (input_bins - 1) // 2
    rho_windows = generate_windows(rho, window_bins).reshape(rho.shape[0], input_bins, 1)
    
    c1_result = model.predict_on_batch([rho_windows, T]).flatten()

    # calculate delta mu
    mu_R = np.log(rho) - c1_result # beta mu
    if eos=="johnson" and rc == 2.5:
        mu_LR = np.array([calc_mu(i, T[0], sig=1, rc="J2.5") for i in rho]).reshape(mu_R.shape)
    else:
        mu_LR = np.array([calc_mu(i, T[0], sig=1, rc=rc) for i in rho]).reshape(mu_R.shape)
    delta_mu = mu_LR - T[0]*mu_R

    return c1_result - delta_mu / T[0]


def Fexc(model, rho, T, dx=0.005):
    """
    Calculate the excess free energy Fexc for a given density profile with functional line integration.

    model: The neural correlation functional
    rho: The density profile
    T: Temperature
    dx: The discretization of the input layer of the model
    """
    alphas = np.linspace(0, 1, 50)
    integrands = np.empty_like(alphas)
    for i, alpha in enumerate(alphas):
        integrands[i] = np.sum(rho * c1_pred(model, alpha * rho, T)) * dx
    Fexc = -simpson(integrands, x=alphas)
    return Fexc

def Fexc_LR(model, rho, T, dx=0.005, rc=2.5):
    """
    Calculate the excess free energy Fexc for a given density profile with functional line integration
    for a long range system using LMFT.

    model: The neural correlation functional
    rho: The bulk density profile
    T: Temperature
    dx: The discretization of the input layer of the model
    """
    alphas = np.linspace(0, 1, 50)
    integrands = np.empty_like(alphas)
    
    for i, alpha in enumerate(alphas):
        c1_sr = c1_pred(model, alpha*rho, T)
        mu_R = np.log(alpha*rho) - c1_sr 
        mu_LR = calc_mu(alpha*np.mean(rho), T, sig=1, rc=rc)
        mu_correction = mu_LR - T*mu_R
        if alpha == 0:
            mu_correction = 0

        c1_lr = c1_sr - mu_correction/T
        integrands[i] = np.sum(rho * c1_lr) * dx
    
    Fexc = -simpson(integrands, x=alphas)
    return Fexc, c1_lr


def binodal_construct(model, T_range, rc=2.5, eos="pets"): # or "johnson", but will automatically use johnson if rc != 2.5
    """
    Generate a binodal from Maxwell construction.

    model: The neural correlation functional
    T_range: Range of temperatures to generate binodal for
    rc: Lennard-Jones cutoff
    eos: Default is "pets" for rc = 2.5 (will use Johnson otherwise) - here in case you wish to use Johnson for rc = 2.5.
    """
    
    # range of rho to sample across
    rho_range = np.concatenate((np.logspace(-3, -1, num=30, base=10), np.linspace(0.11, 0.9, 50)))
    
    rho_g = np.empty_like(T_range)
    rho_l = np.empty_like(T_range)
    mu_co = np.empty_like(T_range)

    for j in range(len(T_range)):
    
        mu = np.empty_like(rho_range)

        # For each temperature, generate a mu-rho equation of state
        for i in range(len(rho_range)):
    
            rho_array = np.ones((2000)) * rho_range[i]
            c1 = c1_pred_lr(model, rho_array, T_range[j], rc=rc, eos=eos) # -beta mu_ex
            mu[i] = np.log(rho_range[i]) - np.mean(c1)

        # From the equation of state generated, perform a Maxwell construction to obtain the coexistence densities

        roots, mu_coex, areas = maxwell.extract_rho_coex(rho_range, mu)
        mu_co[j] = mu_coex
        roots.sort()
        rho_g[j] = roots[0]
        rho_l[j] = roots[-1]

        """
        # Plot the Maxwell constructions if you really want to
        rho1, rho2, mu1, mu2 = areas
        if j ==len(T_range)-1: 
            plt.figure(figsize=(3, 2.5))

            plt.plot(rho_range, mu, color="C{}".format(j))
            plt.scatter([rho_g[j], rho_l[j]], [mu_coex, mu_coex], color="C{}".format(j))
            plt.fill(rho1, mu1, color="C{}".format(j), alpha=0.3)
            plt.fill(rho2, mu2, color="C{}".format(j), alpha=0.3)

            plt.xlabel("$ \\rho_{b} \\sigma^3 $", fontsize=8)
            plt.ylabel("$ \\beta \\mu $", fontsize=8)

            #plt.savefig("../maxwell_construction.png", dpi=600, bbox_inches="tight", pad_inches=0.01)
            plt.show()
        """
        
    return rho_g, rho_l, mu_co
        

## Get Binodal and Vapour Pressure

In [ ]:
dx = 0.005
A = 100 # Lateral system area
kB = 1

model_path = "../../models/WCA.keras"
model = keras.models.load_model(model_path)

#### $r_c = 2.5 \sigma $

In [ ]:
T_range_pets = np.linspace(0.67, 1.088, num=50)
rho_g_pets, rho_l_pets, mu_coex_pets = binodal_construct(model, T_range_pets, rc=2.5)

In [ ]:
#np.savetxt("../data/binodal_rc_2.5.dat", np.c_[T_range_pets, rho_g_pets, rho_l_pets, mu_coex_pets],
#            delimiter=" ", header="temperature rho_g rho_l mu_coex", comments="", fmt=['%.5f', '%.20f', '%.20f', '%.20f'])

In [ ]:
Fexc_range_g = np.empty_like(T_range_pets)
derivPhi_g = np.empty_like(T_range_pets)
z_range = np.ones((2000))
P_g = np.empty((len(T_range_pets)))

for j in range(len(T_range_pets)):
    rho_array_g = z_range * rho_g_pets[j]
    Fex_g, c1_g = Fexc_LR(model, rho_array_g, T_range_pets[j], dx=dx)
    Fexc_range_g[j] = kB * T_range_pets[j] * Fex_g
    derivPhi_g[j] = -np.mean(c1_g) * kB * T_range_pets[j]
    P_g[j] = (derivPhi_g[j] + kB*T_range_pets[j]) * rho_g_pets[j] - A*Fexc_range_g[j]/1000

In [ ]:
#np.savetxt("../data/vapour_pressure_rc_2.5.dat", np.c_[T_range, P_g],
#            delimiter=" ", header="temperature pressure", comments="", fmt=['%.5f', '%.20f'])

#### Full LJ

In [22]:
T_range_full = np.linspace(0.67, 1.3, num=50)
rho_g_full, rho_l_full, mu_coex_full = binodal_construct(model, T_range_full, rc=None)

In [ ]:
#np.savetxt("../data/binodal_rc_full.dat", np.c_[T_range, rho_g_full, rho_l_full, mu_coex_full],
#            delimiter=" ", header="temperature rho_g rho_l mu_coex", comments="", fmt=['%.5f', '%.20f', '%.20f', '%.20f'])

In [ ]:
Fexc_range_g = np.empty_like(T_range_full)
derivPhi_g = np.empty_like(T_range_full)
z_range = np.ones((2000))
P_g_full = np.empty((len(T_range_full)))

for j in range(len(T_range_full)):
    rho_array_g = z_range * rho_g_full[j]
    Fex_g, c1_g = Fexc_LR(model, rho_array_g, T_range_full[j], dx=dx, rc=None)
    Fexc_range_g[j] = kB * T_range_full[j] * Fex_g
    derivPhi_g[j] = -np.mean(c1_g) * kB * T_range_full[j]
    P_g_full[j] = (derivPhi_g[j] + kB*T_range_full[j]) * rho_g_full[j] - A*Fexc_range_g[j]/1000

In [ ]:
#np.savetxt("../data/vapour_pressure_rc_full.dat", np.c_[T_range, P_g_full],
#            delimiter=" ", header="temperature pressure", comments="", fmt=['%.5f', '%.20f'])

## Load Data

#### Simulation Data and PeTS

In [24]:
# PeTS equation of state setup

epsilon_k = 1.0 * si.KELVIN
sigma = 1.0 * si.ANGSTROM

pets = EquationOfState.pets(PetsParameters.from_values(sigma=sigma/si.ANGSTROM, epsilon_k=epsilon_k/si.KELVIN))

dia = PhaseDiagram.pure(
    eos=pets, 
    min_temperature=0.67*epsilon_k, 
    npoints=500
)

In [ ]:
df_lj = pd.read_csv("../data/simulation/binodal_lj_rc_2.5.dat", sep="\t")

T_lj, rho_low_lj, rho_high_lj = zip(*sorted(zip(df_lj["Temperature"].to_numpy(), df_lj["Min"].to_numpy(), df_lj["Max"].to_numpy())))

cut = -3 # Some higher T results are supercritical region

plt.figure(figsize=(5, 4))

plt.plot(dia.liquid.density*si.NAV*sigma**3, dia.liquid.temperature/epsilon_k, color="steelblue", label = "PeTS", zorder=0, linewidth=1.5, linestyle="--", alpha=0.5)
plt.plot(dia.vapor.density*si.NAV*sigma**3, dia.liquid.temperature/epsilon_k, color="steelblue", zorder=1, linewidth=1.5, linestyle="--", alpha=0.5)


plt.scatter(rho_low_lj[:cut], T_lj[:cut], edgecolors="steelblue", label="Sim", zorder=3, marker="s", facecolor="None", linewidth=1.5)
plt.scatter(rho_high_lj[:cut], T_lj[:cut], edgecolors="steelblue", zorder=4, marker="s", facecolor="None", linewidth=1.5)

plt.ylim(top=1.14)

plt.ylabel("$k_B T / \\epsilon$")
plt.xlabel("$\\rho \\sigma^3$")

plt.legend(frameon=False)
plt.tight_layout()

plt.show()

#### Reference Data

In [26]:
##### full LJ binodal #####
### data from https://doi.org/10.1080/00268976.2019.1699185

full_md_sim_T = np.arange(0.69, 1.29, step=0.05)
full_md_sim_rho_v = [0.00164, 0.00320, 0.00522, 0.0087, 0.0133, 0.0198, 0.0276, 0.0380, 0.052, 0.070, 0.094, 0.128, 0.189]
full_md_sim_rho_v_err = [20e-4, 40e-4, 40e-4, 6e-4, 7e-4, 9e-4, 10e-3, 10e-3, 1e-3,2e-3,2e-3,3e-3,4e-3]
full_md_sim_rho_l = [0.847, 0.826, 0.804, 0.781, 0.758, 0.7327, 0.7065, 0.6782, 0.6478, 0.6134, 0.5734, 0.5240, 0.4521]
full_md_sim_rho_l_err = [12e-3, 12e-3,11e-3,11e-3,10e-3,90e-4,80e-4,70e-4,80e-4,70e-4,60e-4,60e-4,40e-4]

In [27]:
##### 2.5 LJ vapour pressure #####
# reference data doi:10.1080/00268970600556774

md_sim_T = np.arange(0.64, 1.06, step=0.03)
md_sim_p = [0.00217, 0.00335, 0.00479, 0.00697, 0.00944, 0.01241, 0.01640,
            0.0214, 0.0274, 0.0336, 0.0417, 0.0504, 0.0606, 0.073, 0.0855]
md_sim_err = [4e-5, 4e-5, 4e-5,4e-5,8e-5,8e-5,8e-5,
              2e-4,2e-4,3e-4,3e-4,4e-4,8e-4,1e-4,8e-4,]

##### full LJ vapour pressure #####
### data from https://doi.org/10.1080/00268976.2019.1699185

md_sim_p_full = [0.00110, 0.00230, 0.00395, 0.0069, 0.0108, 0.0164, 0.0233, 0.0323, 0.0437, 0.0572, 0.0736, 0.0931, 0.1164]
md_sim_full_err = [10e-5, 10e-5, 10e-5,  2e-4, 1e-4, 1e-4, 3e-4, 2e-4, 1e-4, 4e-4, 7e-4, 5e-4, 3e-4]

#### Results

In [30]:
cdft_results_short = pd.read_csv("../data/results/binodal_rc_2.5.dat", sep=" ")
cdft_results_full = pd.read_csv("../data/results/binodal_rc_full.dat", sep=" ")

T_range_pets = cdft_results_short["temperature"].to_numpy()
rho_g_pets = cdft_results_short["rho_g"].to_numpy()
rho_l_pets = cdft_results_short["rho_l"].to_numpy()

T_range_full = cdft_results_full["temperature"].to_numpy()
rho_g_full = cdft_results_full["rho_g"].to_numpy()
rho_l_full = cdft_results_full["rho_l"].to_numpy()

P_g = pd.read_csv("../data/results/vapour_pressure_rc_2.5.dat", sep=" ")["pressure"].to_numpy()
P_g_full = pd.read_csv("../data/results/vapour_pressure_rc_full.dat", sep=" ")["pressure"].to_numpy()

## Plot Figures

In [31]:
def binodal_fit(T, Tc, beta, a, b, rho_c):
    rho_plus = a*abs(T - Tc) + b*abs(T - Tc)**beta + rho_c
    rho_sub = a*abs(T - Tc) - b*abs(T - Tc)**beta + rho_c
    return np.concatenate((rho_sub, rho_plus))

In [ ]:
fig, axs = plt.subplots(ncols=2, nrows=1, layout="compressed", figsize=(6,3), width_ratios=[2.5,4])

##### binodal #####

rho_combined = np.concatenate((rho_low_lj[11:], rho_high_lj[11:]))

initial_guess = [1.188, 0.32630, 0.182, 0.523, 0.320]
parameters = ["Tc", "beta", "a", "b", "rho_c"]

params_opt, params_cov = curve_fit(binodal_fit, T_lj[11:], rho_combined, p0=initial_guess)
perr = np.sqrt(np.diag(params_cov))

temp_fit = np.linspace(0.67, params_opt[0], num=500)
combined_rho_fit = binodal_fit(temp_fit, *params_opt)

axs[1].scatter(rho_low_lj[:cut], T_lj[:cut], edgecolors="steelblue", label="Simulation", zorder=3, marker="s", facecolor="white", linewidth=1.5, s=20)
axs[1].scatter(rho_high_lj[:cut], T_lj[:cut], edgecolors="steelblue", zorder=3, marker="s", facecolor="white", linewidth=1.5, s=20)

##### PeTS rc=2.5 #####

axs[1].plot(rho_g_pets, T_range_pets, label="cDFT + PeTS", color="steelblue", lw=2, zorder=4)
axs[1].plot(rho_l_pets, T_range_pets, color="steelblue",lw=2, zorder=4)

rho_combined = np.concatenate((rho_g_pets[25:], rho_l_pets[25:]))
params_opt, params_cov = curve_fit(binodal_fit, T_range_pets[25:], rho_combined, p0=initial_guess)
perr = np.sqrt(np.diag(params_cov))

print("Critical Exponents for rc=2.5")
for i in range(len(initial_guess)):
    print("{} = {} +/- {}".format(parameters[i], params_opt[i], perr[i]))

axs[1].scatter(params_opt[-1], params_opt[0], color="steelblue", s=25)

temp_fit = np.linspace(0.9, params_opt[0], num=100)
combined_rho_fit = binodal_fit(temp_fit, *params_opt)
axs[1].plot(combined_rho_fit[:len(temp_fit)], temp_fit, color="steelblue", zorder=0, linestyle="dashed")
axs[1].plot(combined_rho_fit[len(temp_fit):], temp_fit, color="steelblue", label="cDFT Fit", zorder=0, linestyle="dashed")

##### full LJ #####

axs[1].errorbar(full_md_sim_rho_v, full_md_sim_T, xerr=full_md_sim_rho_v_err, color="black", label="Stephan and Hasse", fmt="s",
             linewidth=1.5, zorder=1, markerfacecolor="white", markersize=4)
axs[1].errorbar(full_md_sim_rho_l, full_md_sim_T, xerr=full_md_sim_rho_l_err, color="black", fmt="s",
             linewidth=1.5,  zorder=1, markerfacecolor="white",  markersize=4)

axs[1].plot(rho_g_full[:-1], T_range_full[:-1], color="black", lw=2, alpha=0.8, zorder=2)
axs[1].plot(rho_l_full[:-1], T_range_full[:-1], color="black",lw=2, alpha=0.8, zorder=2)

initial_guess = [1.3, 0.32630, 0.182, 0.523, 0.320]
rho_combined = np.concatenate((rho_g_full[35:45], rho_l_full[35:45]))
params_opt, params_cov = curve_fit(binodal_fit, T_range_full[35:45], rho_combined, p0=initial_guess)
perr = np.sqrt(np.diag(params_cov))

print("Critical Exponents for full LJ")
for i in range(len(initial_guess)):
    print("{} = {} +/- {}".format(parameters[i], params_opt[i], perr[i]))

axs[1].scatter(params_opt[-1], params_opt[0], color="black", s=25)

temp_fit = np.linspace(1.1, params_opt[0], num=100)
combined_rho_fit = binodal_fit(temp_fit, *params_opt)
axs[1].plot(combined_rho_fit[:len(temp_fit)], temp_fit, color="black", zorder=5, linestyle="dashed")
axs[1].plot(combined_rho_fit[len(temp_fit):], temp_fit, color="black", label="cDFT Fit", zorder=6, linestyle="dashed")



##### vapour pressure ###

### 2.5 sigma LJ ###

axs[0].errorbar(md_sim_T, md_sim_p, yerr=md_sim_err, color="steelblue", label="Vrabec $et$ $al.$", fmt="p",
             linewidth=1.2,  zorder=0, markersize=5)

axs[0].set_xlabel("$k_B T / \\epsilon$")
axs[0].set_ylabel("$\\beta P \\sigma^3$")

axs[0].plot(T_range_pets, P_g, color="steelblue", linewidth=1.2, zorder=3)

##### full LJ #####

axs[0].errorbar(full_md_sim_T, md_sim_p_full, yerr=md_sim_full_err, color="black", fmt="s",
             linewidth=1.2, markerfacecolor="white", zorder=0, markersize=5)

axs[0].plot(T_range_full, P_g_full, color="black", linewidth=1.2, zorder=3, label="cDFT + Johnson $et$ $al.$")

axs[0].set_xlim(0.6, 1.34)
axs[1].set_ylim(top=1.34)
axs[1].set_ylabel("$k_B T / \\epsilon$")
axs[1].set_xlabel("$\\rho \\sigma^3$")


## Get a combined legend in a specified order
handles, labels = axs[1].get_legend_handles_labels()
handles2, labels2 = axs[0].get_legend_handles_labels()
handles = handles + handles2
labels = labels + labels2
order1 = [0, 6, 1, 2]
order2 = [4, 5, 3]

leg1 = fig.legend([handles[idx] for idx in order1],[labels[idx] for idx in order1] ,bbox_to_anchor=[0.5,1.08], frameon=False, ncols=4, loc="lower center")
leg2 = fig.legend([handles[idx] for idx in order2],[labels[idx] for idx in order2], bbox_to_anchor=[0.5,0.99], frameon=False, ncols=3, loc="lower center")

axs[0].annotate("$r_c = 2.5 \sigma$", (0.75, 0.09), color="steelblue")
axs[0].annotate("Full LJ", (1.05, 0.114))

#fig.savefig("../binodal.png", dpi=600, bbox_inches='tight', pad_inches=0.01, bbox_extra_artists=(leg1, leg2))
plt.show()